# 19 AGEING: before and after the overhaul, and transfer to July

The telemetry month follows the annual overhaul (1 to 29 May 2025), so this notebook asks how the fault process differs across the maintenance cycle.

1. **Fault log before and after the overhaul.** Onsets per in-service day and message mix, 4 November 2024 to 30 April 2025 against 30 May to 21 July 2025, for the messages other than the blower message (which begins on 31 January 2025), with all messages for reference. June and July are also compared, as the first and second months back in service.
2. **Transfer to July.** The primary XGBoost specification is fitted on the June training partition, then scored without refitting on the June test partition and on 1 to 12 July 2025, labelled under the same rules (`03_LABELS`). Pooled and other-message targets, four windows, five seeds. Because July has a lower positive rate, lift over each period's own prevalence is the comparison that matters.

Output: `OUTPUT/v4/rev_ageing_*.csv`. The category shares printed here use an earlier, six-category grouping; the paper reports them with the seven categories of Table 1.

Paper: Section 4.5.

In [ ]:
import os, sys, time, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

V = "v4"            # data version: corrected log with the explicit exclusion rule (REV_REBUILD_v4)
env = os.environ.get
A1, A2 = "../ANALYSIS1", "../ANALYSIS2"
BUILD, DATA = env("REV_BUILDDIR", "build"), env("REV_DATADIR", "DATA")
CFG = dict(
    fault_log = f"{DATA}/37011FL{V}.csv",
    episodes  = f"{BUILD}/fault_episodes_{V}.csv",
    labelled  = f"{BUILD}/telemetry_labelled_{V}.pkl",
    july      = f"{BUILD}/telemetry_labelled_july_{V}.pkl",
    geo       = f"{A1}/DATA/37011/37011GEO.csv",
    outdir    = env("REV_OUTDIR", "OUTPUT") + f"/{V}",
    blower    = "AUX CAP MACH RM BLOW1 NOT OFF",
    june      = ("2025-06-01 00:00:00", "2025-06-30 23:59:59"),
    windows   = {"D5": (-5, 5), "W5": (-5, -1), "W60": (-60, -1), "W300": (-300, -1)},
    seeds     = [int(s) for s in env("REV_SEED_LIST", "42,1,2,3,4").split(",")],
    patience  = 100, test_frac = 0.20, val_frac = 0.20,
)
CATEGORY = {0: "Ventilation", 2: "Protection system", 3: "Protection system",
            5: "Traction and power conversion", 6: "Traction and power conversion",
            7: "Traction and power conversion", 9: "Braking and pneumatic",
            10: "Protection system", 11: "Protection system",
            12: "Traction and power conversion", 13: "Protection system",
            14: "Traction and power conversion", 15: "Auxiliary",
            17: "Power supply quality", 19: "Power supply quality",
            16: "Auxiliary", 8: "Protection system"}   # restored in v4: ASCU messages; Fire Alarm
NUMERIC = ["KMH", "DIST_M", "KV", "AMP", "KWH", "BV", "PRESSURE",
           "TRACTION_PCT", "BG1_KN", "BG2_KN", "BG1_DCLV", "BG2_DCLV"]
WHEEL_SLIP = ["Wheel_Slip_TM2", "Wheel_Slip_TM3", "Wheel_Slip_TM4",
              "Wheel_Slip_TM5", "Wheel_Slip_TM6"]
PRIMARY = NUMERIC + WHEEL_SLIP
os.makedirs(CFG["outdir"], exist_ok=True)
print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| data", V)

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import average_precision_score

def chrono_cut(y, frac):
    n = len(y); i = int(n * (1 - frac))
    while i < n - 1 and y[i] == 1 and y[i - 1] == 1:
        i += 1
    return i

def three_way_split(y):
    n = len(y); t = int(n * (1 - CFG["test_frac"]))
    while t < n - 1 and y[t] == 1:
        t += 1
    v = chrono_cut(y[:t], CFG["val_frac"])
    return np.arange(v), np.arange(v, t), np.arange(t), np.arange(t, n)

def xgb(y, seed, n_iter=None):
    spw = float((y == 0).sum() / max((y == 1).sum(), 1))
    kw = dict(n_estimators=600, early_stopping_rounds=CFG["patience"]) if n_iter is None else dict(n_estimators=n_iter)
    return XGBClassifier(learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8,
                         eval_metric="aucpr", tree_method="hist", scale_pos_weight=spw,
                         random_state=seed, n_jobs=-1, **kw)

def fit_primary(X, y, seed):
    """Two-stage fit of 07_CLASSIFY (default class weighting). Returns the model fitted on the
    training partition, the test indices and the number of trees."""
    ia, iv, itr, ite = three_way_split(y)
    m1 = xgb(y[ia], seed).fit(X[ia], y[ia], eval_set=[(X[iv], y[iv])], verbose=False)
    n_iter = int(m1.best_iteration) + 1
    return xgb(y[itr], seed, n_iter).fit(X[itr], y[itr]), ite, n_iter

def pr(y, s):
    base = y.mean()
    ap = average_precision_score(y, s) if y.sum() > 0 else np.nan
    return dict(pr_auc=ap, baseline=base, lift=ap / base if base > 0 else np.nan, positives=int(y.sum()))

def coverage(ts, onsets, lo, hi):
    d = np.zeros(len(ts) + 1, dtype=np.int32)
    o = pd.DatetimeIndex(onsets).values.astype("datetime64[s]")
    a = np.searchsorted(ts, o + np.timedelta64(lo, "s"), side="left")
    b = np.searchsorted(ts, o + np.timedelta64(hi, "s"), side="right")
    np.add.at(d, a, 1); np.add.at(d, b, -1)
    return np.cumsum(d)[:-1]

def restrict(y0, ts, keep_onsets, other_onsets, h):
    """Positives produced only by onsets outside the kept set are excluded (-1), as in 11_CHECK_BLOWER."""
    lo, hi = CFG["windows"][h]
    cov_k = coverage(ts, keep_onsets, lo, hi)
    cov_o = coverage(ts, other_onsets, lo, hi)
    return np.where((y0 == 1) & (cov_o > 0) & (cov_k == 0), -1, y0)

## 1. Fault log before and after the overhaul

In [ ]:
from scipy.stats import mannwhitneyu
ep = pd.read_csv(CFG["episodes"], parse_dates=["onset", "recovery"])
ep["blower"] = ep.fault.eq(CFG["blower"])
g = ep.groupby(pd.Grouper(key="onset", freq="D"))
d = pd.DataFrame({"all": g.size(), "other": g["blower"].apply(lambda s: int((~s).sum()))})
d = d.asfreq("D").fillna(0)
d.loc["2025-05-01":"2025-05-29"] = np.nan          # out of service (annual overhaul)
PERIODS = {"before (4 Nov - 30 Apr)": ("2024-11-04", "2025-04-30"),
           "after (30 May - 21 Jul)": ("2025-05-30", "2025-07-21"),
           "June": ("2025-06-01", "2025-06-30"), "July 1-21": ("2025-07-01", "2025-07-21")}
rows = []
for name, (a, b) in PERIODS.items():
    s = d.loc[a:b].dropna()
    rows.append(dict(period=name, days=len(s), all_per_day=s["all"].mean(), other_per_day=s["other"].mean(),
                     other_median=s["other"].median(), other_days_zero=int((s["other"] == 0).sum())))
rate = pd.DataFrame(rows).set_index("period")
print(rate.round(2).to_string())
b4, af = d.loc["2024-11-04":"2025-04-30", "other"].dropna(), d.loc["2025-05-30":"2025-07-21", "other"].dropna()
u = mannwhitneyu(b4, af)
print(f"\nOther labels, daily onsets before vs after: Mann-Whitney p = {u.pvalue:.3f} "
      f"(days are serially dependent, so the p value is indicative only)")
rate.to_csv(f"{CFG['outdir']}/rev_ageing_rates.csv")

ep["period"] = np.where(ep.onset < "2025-05-01", "before", np.where(ep.onset >= "2025-05-30", "after", "shed"))
o = ep[~ep.blower & ep.period.isin(["before", "after"])]
o["category"] = o.cluster.map(CATEGORY).fillna("unassigned")
mix = pd.crosstab(o.category, o.period, normalize="columns").mul(100).round(1)[["before", "after"]]
lab = pd.crosstab(o.fault, o.period, normalize="columns").mul(100).round(1)[["before", "after"]]
lab = lab.loc[lab.max(axis=1).sort_values(ascending=False).index[:12]]
print("\nOther labels, share by category (%)\n", mix.to_string())
print("\nOther labels, leading labels (%)\n", lab.to_string())
pd.concat({"category": mix, "label": lab}).to_csv(f"{CFG['outdir']}/rev_ageing_mix.csv")

## 2. Transfer of the June model to July

In [ ]:
tel = pd.read_pickle(CFG["labelled"]).sort_values("DATETIME").reset_index(drop=True)
jul = pd.read_pickle(CFG["july"]).sort_values("DATETIME").reset_index(drop=True)
ts_j, ts_l = (t["DATETIME"].values.astype("datetime64[s]") for t in (tel, jul))
on_j = ep[ep.onset.between(*CFG["june"])]
on_l = ep[ep.onset.between("2025-07-01", "2025-07-12 23:59:59")]
Xj = tel[PRIMARY].astype("float32").fillna(0).values
Xl = jul[PRIMARY].astype("float32").fillna(0).values
print(f"June {len(tel):,} records, {len(on_j)} onsets; July 1-12 {len(jul):,} records, {len(on_l)} onsets")

def targets(t, ts, on, h):
    y0 = t[h].astype(int).values
    return {"pooled": y0,
            "other labels": restrict(y0, ts, on.onset[~on.blower], on.onset[on.blower], h)}

rows = []
t_all = time.time()
for h in env("REV_HORIZONS", "D5,W5,W60,W300").split(","):
    tj, tl = targets(tel, ts_j, on_j, h), targets(jul, ts_l, on_l, h)
    for kind in ["pooled", "other labels"]:
        kj, kl = tj[kind] != -1, tl[kind] != -1
        for seed in CFG["seeds"]:
            m, ite, n_iter = fit_primary(Xj[kj], tj[kind][kj], seed)
            rj = pr(tj[kind][kj][ite], m.predict_proba(Xj[kj][ite])[:, 1])
            rl = pr(tl[kind][kl], m.predict_proba(Xl[kl])[:, 1])
            for per, r in [("June test", rj), ("July 1-12", rl)]:
                r.update(target=kind, horizon=h, seed=seed, period=per, trees=n_iter); rows.append(r)
            print(f"{h:5s} {kind:12s} seed {seed:2d}  June PR-AUC {rj['pr_auc']:.3f} (lift {rj['lift']:.1f})   "
                  f"July {rl['pr_auc']:.3f} (lift {rl['lift']:.1f})  trees {n_iter}", flush=True)
res = pd.DataFrame(rows)
res.to_csv(f"{CFG['outdir']}/rev_ageing_transfer.csv", index=False)
print(f"[{(time.time() - t_all) / 60:.1f} min]")

## 3. Summary over seeds

In [ ]:
s = (res.groupby(["target", "horizon", "period"])
        .agg(baseline=("baseline", "first"), positives=("positives", "first"),
             pr_auc_mean=("pr_auc", "mean"), pr_auc_sd=("pr_auc", "std"), lift_mean=("lift", "mean"))
        .unstack("period"))
s = s.reindex([h for h in ["D5", "W5", "W60", "W300"] if h in res.horizon.unique()], level="horizon")
print(s.round(4).to_string())
s.to_csv(f"{CFG['outdir']}/rev_ageing_transfer_summary.csv")

___
**END**